This notebook is similar to the previous one, 3_BooksCosineSimilarity.ipynb, but instead of similar scores given by identical group of reviewers, this notebook gathers user's text review and summary, vectorize this textual data and computes the cosine similarity between books. The most similar books are then recommended based on the text or summary user review. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import kagglehub, os
import matplotlib.pyplot as plt 
import seaborn as sns
from scipy.sparse import csr_matrix

C:\Users\yonau\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Amazon books review dataset 
path = kagglehub.dataset_download("mohamedbakhet/amazon-books-reviews")
print("Path to dataset files:", path)
for file in os.listdir(path):
    print(file)

Path to dataset files: C:\Users\yonau\.cache\kagglehub\datasets\mohamedbakhet\amazon-books-reviews\versions\1
books_data.csv
Books_rating.csv


In [3]:
books_data = pd.read_csv(os.path.join(path, "books_data.csv"))
rating_data = pd.read_csv(os.path.join(path, "Books_rating.csv"))

In [4]:
def clean_data(rating_data, col_name): 
    assert col_name in ['review/summary', 'review/text'], "Please the column name must be 'review/text' or 'review/summary'."
    # Keeping books with + than 10 rates 
    ratescores = rating_data.dropna(subset=['User_id', 'Title', col_name]).copy()
    # Making data lighter on memory - possible to adjust numbers for larger memory 
    # Keeping only active raters meaning users who rated more than 5 books 
    user_counts = ratescores['User_id'].value_counts()
    ratescores = ratescores[ratescores['User_id'].isin(user_counts[user_counts >= 10].index)]
    book_counts = ratescores['Title'].value_counts()
    ratescores = ratescores[ratescores['Title'].isin(book_counts[book_counts >= 20].index)] 
    # Grouping all review/text or summary col in one cell for each Book 
    # concat all reviews for a global user's review book similarity evaluation 
    print("Text review concatenation...")
    book_profiles = ratescores.groupby('Title')[col_name].apply(lambda texts: ' '.join(texts.astype(str))).reset_index()
    print(f"Number of unique books: {len(book_profiles)}")
    return book_profiles, ratescores

In [5]:
# TF-IDF Initialization 
# TF-IDF stands for term frequency-inverse document frequency, which is a statistical measure 
# used to evaluate the importance of a word in a document relative to a collection of documents. 
# It combines two components: term frequency (how often a word appears in a document) and 
# inverse document frequency (how unique or rare that word is across all documents)
# Convert a collection of raw text to a matrix of TF-IDF features.
def compute_TF_IDF_vect(book_review_profiles, col_name):
    tfidf = TfidfVectorizer(
        stop_words='english', 
        max_features=10000,  # keeping 10 000 most frequent words from sequence
        ngram_range=(1, 2)   # can optionaly capture 2 gram words in case of bigram combinaison 
    )
    print("Computing TF-IDF matrix...")
    tfidf_matrix = tfidf.fit_transform(book_review_profiles[col_name])
    return tfidf_matrix

In [6]:
def recommend(book_title, ratescores_df, sim_matrix, top_n=5):
    """
    Recommends top books for a given title, based on similar given scores. 
    """
    # retrieve previous given indexes
    unique_titles = ratescores_df['Title'].astype('category').cat.categories
    if book_title not in unique_titles:
        return f"The book'{book_title}'has not enough reviews or has been filtred and does not appear in the database currently."

    idx = unique_titles.get_loc(book_title) # get the book location
    sim_scores = list(enumerate(sim_matrix[idx])) # get the line corresponding to this book in the similarity matrix 
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) # score order 
    sim_scores = sim_scores[1:top_n+1] # index 0 is the book itself 

    book_indices = [i[0] for i in sim_scores]
    recommendations = [unique_titles[i] for i in book_indices] # reassociate book titles 
    
    return recommendations

In [11]:
for col_name in ['review/text', 'review/summary']: 
    book_review_profiles, ratescores = clean_data(rating_data, col_name)
    tfidf_matrix = compute_TF_IDF_vect(book_review_profiles, col_name)
    # Compute siilarity matrix 
    cosine_sim_text = cosine_similarity(tfidf_matrix)
    print(f"With {col_name}, recommendations are:")
    print(recommend("My Life", ratescores, cosine_sim_text))


Text review concatenation...
Number of unique books: 5429
Computing TF-IDF matrix...
With review/text, recommendations are:
['The Natural: The Misunderstood Presidency of Bill Clinton', 'The Final Days', 'The Case Against Hillary Clinton', "The Truth about Hillary (Clinton): What She Knew, When She Knew It, and How Far She'll Go to Become President", "Losing Bin Laden: How Bill Clinton's Failures Unleashed Global Terror"]
Text review concatenation...
Number of unique books: 5429
Computing TF-IDF matrix...
With review/summary, recommendations are:
["Losing Bin Laden: How Bill Clinton's Failures Unleashed Global Terror", 'The Natural: The Misunderstood Presidency of Bill Clinton', 'The Case Against Hillary Clinton', 'The Final Days', 'Truman']


In [12]:
recommend("1984", ratescores, cosine_sim_text)

['George Orwell 1984',
 'Nineteen Eighty-four',
 'Animal Farm and 1984',
 "Animal Farm & 1984 (Collected Stories of the World's Greatest Writers)",
 'George Orwell: Animal Farm-Nineteen Eighty-Four']